# ECERC Limitation 2, follow-up: do the text emotion features carry the training labels?

The first experiment showed that ECERC's rare-emotion errors are not a decision-threshold problem. One lead stood out: a simple classifier on the text emotion features fits rare emotions almost perfectly on training data but fails on dev. That would happen if the RoBERTa model that produced these features, fine-tuned on the MELD training utterances, had baked the training labels into them.

This notebook tests that in about 10 minutes, using only the **train and dev** splits; the test set is never read:
1. Linear classifiers are trained on 4/5 of the training dialogues and scored on the held-out 1/5 *and* on dev. Without a leak, the two scores should be close.
2. The same is done with plain RoBERTa features (not fine-tuned) as a control, plus audio and visual features.
3. The pre-registered rule (shown in the output) gives the verdict.

**How to run:** a CPU runtime is enough (Runtime → Change runtime type → CPU), then Runtime → **Run all**. Don't edit cells. If a cell stops with an error, send the error text.

In [ ]:
# ===== Configuration: leave as is. (Set TRUST_DATA_PICKLE = True only if the data cell asks you to.) =====
import os

TRUST_DATA_PICKLE = False

if os.path.isdir('/kaggle/working'):
    WORK_DIR = '/kaggle/working/ecerc_l2'
elif os.path.isdir('/content'):
    WORK_DIR = '/content/ecerc_l2'
else:
    WORK_DIR = os.path.abspath('ecerc_l2_work')
CONFIG = dict(
    work_dir=WORK_DIR,
    data_file_id='1IQokygWgjaZuSXvCsfIcUzj1Gu69hDCb',            # authors' Google Drive (linked from the official ECERC README)
    data_filename='meld_emotion_semantic_features_roberta.pkl',
    expected_data_sha256='a0349614b2b58e8b2db768f0144aa36b96fb2348ca436a6879ee2623cc2d2f80',  # observed in run_colab_1
    trust_data_pickle=TRUST_DATA_PICKLE,
    expected_utterances=dict(train=9989, valid=1109),
    n_folds=5, fold_seed=2026, C=1.0, max_iter=1000,   # the same probe as diagnostic D3 of the first run
    bootstrap_B=2000, boot_seed=777,
    offline=False,                          # True only for the local smoke test
)
os.makedirs(WORK_DIR, exist_ok=True)
print('Working directory:', WORK_DIR)

## 1. Code
The analysis code (`l2_posthoc.py` from the first experiment and `l2_feature_leak.py`), embedded as base64 and checked by SHA-256. You don't need to read it. Readable copies are in `research/limitation2/src/`.

In [ ]:
# Writes the analysis code, checks its SHA-256, then sets up.
import base64, hashlib, importlib, sys

CODE_DIR = os.path.join(CONFIG['work_dir'], 'code')
os.makedirs(CODE_DIR, exist_ok=True)
CODE_FILES = {  # name: (sha256, base64 content)
    'l2_posthoc.py': ('...
fSBNQiknKQogICAgcmV0dXJuIHpwYXRoCg==
'''),
    'l2_feature_leak.py': ('...
'''),
}
for name, (digest, blob) in CODE_FILES.items():
    data = base64.b64decode(blob)
    assert hashlib.sha256(data).hexdigest() == digest, f'{name} is corrupted'
    with open(os.path.join(CODE_DIR, name), 'wb') as f:
        f.write(data)
print('Code files written and verified:', ', '.join(CODE_FILES))
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
import l2_posthoc
importlib.reload(l2_posthoc)
import l2_feature_leak as LK
importlib.reload(LK)
S = LK.stage_setup(CONFIG, CODE_DIR)

## 2. Data (the authors' MELD features from Google Drive)
If the download fails, follow the printed manual steps, then **Run all** again.

In [ ]:
LK.stage_download(S)

## 3. Load the train and dev splits (the test split is never read)
If it stops with a *blocked global* message, set `TRUST_DATA_PICKLE = True` in the first code cell and **Run all** again.

In [ ]:
LK.stage_load(S)

## 4. Linear classifiers and nearest neighbours
20 classifiers (4 feature types × 5 folds) run in parallel, which takes a few minutes. Then a nearest-neighbour check.

In [ ]:
LK.stage_probes(S)
LK.stage_nearest_neighbour(S)

## 5. Verdict (pre-registered rule)

In [ ]:
LK.stage_decide(S)

## 6. Results
Prints the summary and downloads `ecerc_l2_leak_results.zip`. If the download doesn't start, open the Files panel (folder icon on the left), find `ecerc_l2/ecerc_l2_leak_results.zip`, and download it from its menu.

In [ ]:
zpath = LK.stage_package(S)
try:
    from google.colab import files
    files.download(zpath)
except ImportError:
    print('Not running on Colab: download this file manually ->', zpath)